# Agent Zero — Fixed Colab Setup

This notebook installs and launches [Agent Zero](https://github.com/agent0ai/agent-zero) in Google Colab
and exposes it to the browser via an ngrok tunnel.

**Root cause of the previous `Internal Server Error` / `CalledProcessError`:**
Earlier setups tried to pin `openai==1.50.2` together with `litellm==1.49.5` or
`litellm==1.50.0`. Both of those `litellm` releases internally import
`PromptTokensDetails` from `openai.types.completion_usage`, a class that does
*not* exist in `openai 1.50.2` (only `CompletionTokensDetails` is present in that
release). When pip is invoked with `subprocess.check_call` and the resolution
fails, Python surfaces a `CalledProcessError` instead of a human-readable message.

**The fix applied here:**
1. Use `importlib.metadata` (not `pkg_resources`) for safe version introspection.
2. Inspect shared `Requires-Dist` for both packages *before* installing to surface
   any constraint mismatch early.
3. Install packages incrementally with `subprocess.run(..., check=False)` and
   explicit return-code checking so any failure prints a clear message instead of
   an unhandled `CalledProcessError`.
4. Let pip choose the latest compatible versions — never hard-pin `openai==1.50.2`
   or `litellm==1.49.5` / `litellm==1.50.0`.

## Step 1 — Clone Agent Zero and upgrade build tools

In [ ]:
%cd /content
!rm -rf agent-zero
!git clone https://github.com/agent0ai/agent-zero.git
%cd agent-zero

!pip install --upgrade pip setuptools wheel -q

## Step 2 — Diagnose the current environment

In [ ]:
import importlib.metadata
import os

libs = ['openai', 'litellm', 'pydantic', 'typing-extensions', 'anyio']
print('--- Installed versions ---')
for lib in libs:
    try:
        print(f"{lib}: {importlib.metadata.version(lib)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{lib}: NOT INSTALLED")

print('\n--- Working directory ---')
print(os.getcwd())

print('\n--- .env presence ---')
print(f"root .env : {os.path.exists('.env')}")
print(f"usr/.env  : {os.path.exists('usr/.env')}")

## Step 3 — Check version availability, inspect constraints, and install

This step implements the three-phase diagnostic strategy:
1. **Check Version Availability** — verify that the required package versions exist on PyPI.
2. **Identify Constraint Mismatch** — compare `Requires-Dist` for shared libraries
   (`pydantic`, `anyio`, `typing-extensions`) to find conflicts before they cause a
   `CalledProcessError`.
3. **Incremental Installation** — install packages one-by-one with explicit error
   handling so the exact failure point is always visible.

In [ ]:
import subprocess
import sys
import json as _json
import importlib.metadata
import urllib.request

# ── Helper: run pip safely (no CalledProcessError) ───────────────────────
def pip_install(*args, label=None):
    """Run pip install and return True on success, False on failure."""
    cmd = [sys.executable, '-m', 'pip', 'install'] + list(args)
    tag = label or ' '.join(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f'\u2705 {tag}')
        return True
    else:
        print(f'\u274c {tag}\n    pip stderr: {result.stderr.strip()}')
        return False

# ── Helper: fetch latest PyPI version ────────────────────────────────────
def pypi_latest(package):
    """Return the latest version string for a package from PyPI."""
    try:
        url = f'https://pypi.org/pypi/{package}/json'
        with urllib.request.urlopen(url, timeout=5) as r:
            return _json.load(r)['info']['version']
    except Exception as exc:
        return f'(lookup failed: {exc})'

# ─────────────────────────────────────────────────────────────────────────
# PHASE 1: Check version availability
# ─────────────────────────────────────────────────────────────────────────
print('=' * 60)
print('PHASE 1 — Version Availability')
print('=' * 60)

# These are the two historically broken pins; we confirm they exist but
# will NOT install them together.
bad_pins = [('openai', '1.50.2'), ('litellm', '1.49.5'), ('litellm', '1.50.0')]
for pkg, ver in bad_pins:
    try:
        url = f'https://pypi.org/pypi/{pkg}/{ver}/json'
        with urllib.request.urlopen(url, timeout=5) as r:
            data = _json.load(r)
        print(f'  {pkg}=={ver} exists on PyPI (confirmed) — do NOT pin these together')
    except Exception:
        print(f'  {pkg}=={ver} NOT found on PyPI')

print()
for pkg in ('openai', 'litellm'):
    latest = pypi_latest(pkg)
    print(f'  Latest {pkg}: {latest}')

# ─────────────────────────────────────────────────────────────────────────
# PHASE 2: Identify constraint mismatch
# ─────────────────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('PHASE 2 — Constraint Inspection (dry-run with --no-deps)')
print('=' * 60)

# Temporarily install both packages without dependencies so we can read
# their metadata without triggering a resolution conflict.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'openai==1.50.2', 'litellm==1.50.0', '--no-deps', '-q'],
    capture_output=True
)

shared_filters = ['pydantic', 'anyio', 'typing-extensions']
for pkg in ('openai', 'litellm'):
    try:
        dist = importlib.metadata.distribution(pkg)
        reqs = importlib.metadata.requires(pkg) or []
        print(f'\n  {pkg} {dist.version} — shared requirements:')
        matched = [r for r in reqs if any(f in r.lower() for f in shared_filters)]
        if matched:
            for r in matched:
                print(f'    {r}')
        else:
            print('    (none of the filtered shared deps found)')
    except importlib.metadata.PackageNotFoundError:
        print(f'  {pkg}: not installed (--no-deps install may have failed)')

# Now uninstall the temporarily installed versions before the real install.
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'openai', 'litellm'],
    capture_output=True
)

# ─────────────────────────────────────────────────────────────────────────
# PHASE 3: Incremental installation
# ─────────────────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('PHASE 3 — Incremental Installation')
print('=' * 60)

# Purge stale cached wheels first.
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], capture_output=True)
print('pip cache purged')

steps = [
    (['--upgrade', 'pip', 'setuptools', 'wheel'], 'build tools'),
    (['openai'],    'openai (latest compatible)'),
    (['litellm'],   'litellm (latest compatible)'),
    (['-r', 'requirements.txt'], 'Agent Zero requirements.txt'),
    (['flask', 'flask-cors', 'pydantic', 'python-dotenv', 'pyngrok'],
     'web-server and tunnel deps'),
]

failed_steps = []
for args, label in steps:
    ok = pip_install(*args, label=label)
    if not ok:
        failed_steps.append(label)

print()
if not failed_steps:
    print('\u2705 All incremental install steps succeeded.')
else:
    print(f'\u26a0\ufe0f The following steps failed: {failed_steps}')
    print('   Review the pip stderr messages above to diagnose the conflict.')

## Step 4 — Verify the installation

In [ ]:
import importlib
import importlib.metadata

errors = []
for pkg in ('openai', 'litellm', 'flask', 'pydantic'):
    try:
        mod = importlib.import_module(pkg)
        ver = importlib.metadata.version(pkg)
        print(f"\u2705 {pkg} {ver}")
    except Exception as exc:
        print(f"\u274c {pkg} — {exc}")
        errors.append(pkg)

# Verify that the specific attribute that previously caused the crash is accessible.
try:
    import litellm  # noqa: F401
    print('\u2705 litellm imported cleanly (no PromptTokensDetails error)')
except Exception as exc:
    print(f'\u274c litellm import failed: {exc}')
    errors.append('litellm-import')

if not errors:
    print('\n\u2705 All packages verified. Ready to launch Agent Zero.')
else:
    print(f'\n\u26a0\ufe0f Issues found in: {errors}. See troubleshooting notes below.')

## Step 5 — Configure environment variables

Edit the cell below to add your API key(s) before running it.

In [ ]:
import os

# ── Set your API keys here ──────────────────────────────────────────────────
ANTHROPIC_API_KEY = ""   # e.g. "sk-ant-..."
OPENAI_API_KEY    = ""   # e.g. "sk-..."
NGROK_AUTH_TOKEN  = ""   # from https://dashboard.ngrok.com/get-started/your-authtoken
# ───────────────────────────────────────────────────────────────────────────

env_lines = []
if ANTHROPIC_API_KEY:
    env_lines.append(f"ANTHROPIC_API_KEY={ANTHROPIC_API_KEY}")
if OPENAI_API_KEY:
    env_lines.append(f"OPENAI_API_KEY={OPENAI_API_KEY}")

# Write to the location Agent Zero expects.
os.makedirs('usr', exist_ok=True)
with open('usr/.env', 'w') as fh:
    fh.write('\n'.join(env_lines) + '\n')

print(f"usr/.env written with {len(env_lines)} key(s).")

## Step 6 — Launch Agent Zero and expose it via ngrok

In [ ]:
import subprocess
import sys
import time
import requests
from pyngrok import ngrok, conf

PORT = 5000

# ── Configure ngrok ────────────────────────────────────────────────────────
if not NGROK_AUTH_TOKEN:
    raise ValueError("Set NGROK_AUTH_TOKEN in Step 5 before running this cell.")

conf.get_default().auth_token = NGROK_AUTH_TOKEN

# ── Start the Agent Zero web server ───────────────────────────────────────
server_proc = subprocess.Popen(
    [sys.executable, 'run_ui.py', '--port', str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(f'Agent Zero process started (PID {server_proc.pid})')

# ── Wait until the server is ready ────────────────────────────────────────
print('Waiting for server to become ready ', end='')
server_ready = False
for _ in range(30):
    time.sleep(1)
    print('.', end='', flush=True)
    if server_proc.poll() is not None:
        # Process exited early — print its output and abort.
        output = server_proc.stdout.read()
        print(f'\n\u274c Server process exited unexpectedly (code {server_proc.returncode}).')
        print('Server output:\n', output)
        break
    try:
        if requests.get(f'http://localhost:{PORT}', timeout=2).status_code < 500:
            print(' ready!')
            server_ready = True
            break
    except Exception:
        pass
else:
    print(' timed out \u2014 check server logs above.')

if not server_ready:
    raise RuntimeError(
        'Agent Zero server failed to start. '
        'Check the server output printed above for Python tracebacks.'
    )

# ── Open ngrok tunnel ─────────────────────────────────────────────────────
tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url

print('=' * 60)
print('\U0001f680 AGENT ZERO IS LIVE!')
print(f'\U0001f517 URL: {public_url}')
print('=' * 60)
print('Open the URL above in your browser to use the Agent Zero UI.')

## Troubleshooting

| Symptom | Cause | Fix |
|---------|-------|-----|
| `cannot import name 'PromptTokensDetails'` | `litellm==1.49.5` or `litellm==1.50.0` expects an attribute absent in `openai==1.50.2` | Re-run **Step 3** — the modern pip-resolved versions are compatible |
| `CalledProcessError` from pip | A `subprocess.check_call` pip invocation failed; the real conflict is hidden in stderr | Step 3 now uses `subprocess.run(check=False)` and prints the stderr directly |
| `ResolutionImpossible` from pip | Pinning `openai==1.50.2` + `litellm==1.49.5` or `litellm==1.50.0` together | Do **not** pin those old versions; let pip choose |
| `Internal Server Error` in the UI | Stale `.pyc` / cached wheels from a previous run | Re-run Phase 3 in **Step 3** (`pip cache purge` is included) |
| ngrok `ERR_NGROK_108` | Auth token missing or expired | Set `NGROK_AUTH_TOKEN` in **Step 5** |
| Server never becomes ready | Flask failed to start | Step 6 now prints the full server stdout on early exit |
| `pkg_resources` / `pkgutil.ImpImporter` error | Old diagnostic code used `pkg_resources` (broken in Python 3.12) | Steps 2 and 3 use `importlib.metadata` throughout |